# Lesson 02: Vector Embeddings & 3D Visualization
> **Goal:** Understand what vector embeddings are, inspect their dimensions, measure semantic similarity, and visualize meaning in 3D.

---

### What is a Vector Embedding?
An embedding model turns text into a list of numbers (a **vector**) where meaning is represented as **geometric coordinates**.

- **The Analogy:** Just as GPS coordinates `(Latitude, Longitude)` describe physical location on Earth, an embedding's coordinates describe location in **semantic concept space**.
- Phrases with similar meanings (e.g. *"puppy"* and *"dog"*) land close together in vector space.
- Phrases with unrelated meanings (e.g. *"puppy"* and *"neural network"*) land far apart.

```mermaid
flowchart LR
    A["Text: 'golden retriever puppy'"] --> B["Embedding Model<br/>(gemini-embedding-001)"]
    B --> C["768 Coordinates<br/>[0.0095, 0.0558, -0.0221, ...]"]
    C --> D["Plot coordinates in space<br/>(Cluster by Meaning!)"]
```


In [16]:
# Step 1: Imports and environment setup
import os
from dotenv import load_dotenv, find_dotenv
import numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from google import genai

# Load environment variables (.env) - searches current and parent folders
load_dotenv(find_dotenv())


True

In [17]:
# Step 2: Define 10 sample sentences across 4 clear topics
texts = [
    # Topic 1: Animals & Pets (Green)
    "playful golden retriever puppy",
    "kitten chasing ball yarn",
    "friendly loyal domestic dog",

    # Topic 2: AI & Technology (Blue)
    "deep learning neural network",
    "artificial intelligence python code",
    "training machine learning models",

    # Topic 3: Food & Cooking (Orange)
    "baking fresh sourdough bread",
    "delicious hot Italian pizza",

    # Topic 4: Space & Astronomy (Purple)
    "distant galaxy deep space",
    "shining stars solar system"
]

print(f"Loaded {len(texts)} sentences across 4 topics.")


Loaded 10 sentences across 4 topics.


In [18]:
# Step 3: Generate 768-dimensional embeddings using Google GenAI SDK
# Uses gemini-embedding-001 (supported by API v1) with 768 dimensions
client = genai.Client()

response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=texts,
    config={"output_dimensionality": 768}
)

# Extract the numeric vector for each sentence
embeddings = [e.values for e in response.embeddings]

print(f"Successfully generated {len(embeddings)} embeddings of dimension {len(embeddings[0])}!")


Successfully generated 10 embeddings of dimension 768!


In [19]:
# Step 4: Inspect the coordinates of each vector
# Each sentence is now represented by 768 continuous floating-point numbers.
print(f"{'Sentence':<36} | {'Dimensions':<10} | Sample Coordinates")
print("-" * 72)
for text, emb in zip(texts, embeddings):
    sample = [round(x, 4) for x in emb[:3]]
    print(f"{text:<36} | {len(emb)} dims   | {sample}...")


Sentence                             | Dimensions | Sample Coordinates
------------------------------------------------------------------------
playful golden retriever puppy       | 768 dims   | [-0.0193, 0.0243, 0.0062]...
kitten chasing ball yarn             | 768 dims   | [-0.0394, 0.0178, -0.0016]...
friendly loyal domestic dog          | 768 dims   | [-0.006, 0.0167, 0.0137]...
deep learning neural network         | 768 dims   | [-0.0238, -0.0021, 0.0176]...
artificial intelligence python code  | 768 dims   | [-0.0096, -0.0045, 0.0102]...
training machine learning models     | 768 dims   | [-0.0147, 0.0114, 0.0068]...
baking fresh sourdough bread         | 768 dims   | [0.0092, 0.0326, 0.0041]...
delicious hot Italian pizza          | 768 dims   | [-0.0153, 0.009, 0.022]...
distant galaxy deep space            | 768 dims   | [-0.0206, 0.0125, 0.004]...
shining stars solar system           | 768 dims   | [0.0038, 0.002, 0.0477]...


In [20]:
# Step 5: Measuring Semantic Similarity (Cosine Similarity)
# How does an AI model tell if two sentences mean the same thing?
# It calculates the angle between their vectors: 1.0 = identical direction.

def cosine_similarity(vec1: list[float], vec2: list[float]) -> float:
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

sim_pets = cosine_similarity(embeddings[0], embeddings[2])  # puppy vs dog
sim_ai_pet = cosine_similarity(embeddings[0], embeddings[3])  # puppy vs neural network
sim_food_space = cosine_similarity(embeddings[7], embeddings[9])  # pizza vs solar system

print("Semantic Similarity (Cosine Similarity):")
print("-" * 70)
print(f"Same Topic  : 'puppy' vs 'dog'             -> {sim_pets:.4f}  (High similarity)")
print(f"Diff Topic  : 'puppy' vs 'neural network'  -> {sim_ai_pet:.4f}  (Lower similarity)")
print(f"Diff Topic  : 'pizza' vs 'solar system'    -> {sim_food_space:.4f}  (Lower similarity)")


Semantic Similarity (Cosine Similarity):
----------------------------------------------------------------------
Same Topic  : 'puppy' vs 'dog'             -> 0.6495  (High similarity)
Diff Topic  : 'puppy' vs 'neural network'  -> 0.4815  (Lower similarity)
Diff Topic  : 'pizza' vs 'solar system'    -> 0.5333  (Lower similarity)


In [21]:
# Step 6: Reduce 768 dimensions down to 3D using PCA
# We cannot visualize 768 dimensions in the physical world.
# PCA finds the 3 most significant directions of variance so we can plot them in 3D.
pca = PCA(n_components=3, random_state=42)
embeddings_3d = pca.fit_transform(embeddings)

print(f"Reduced from {len(embeddings[0])} dimensions down to {embeddings_3d.shape[1]} dimensions (X, Y, Z).")


Reduced from 768 dimensions down to 3 dimensions (X, Y, Z).


In [23]:
# Step 7: Interactive 3D visualization with Plotly
# Color mapping: Green (Pets), Blue (AI), Orange (Food), Purple (Space)
colors = ["#10B981"] * 3 + ["#3B82F6"] * 3 + ["#F59E0B"] * 2 + ["#A855F7"] * 2

fig = go.Figure(data=[go.Scatter3d(
    x=embeddings_3d[:, 0],
    y=embeddings_3d[:, 1],
    z=embeddings_3d[:, 2],
    mode="markers+text",
    text=texts,
    textposition="top center",
    marker=dict(size=7, color=colors)
)])

fig.update_layout(
    title="3D Text Embeddings Visualization (PCA Clusters)",
    scene=dict(
        xaxis=dict(title="PC 1"),
        yaxis=dict(title="PC 2"),
        zaxis=dict(title="PC 3")
    ),
    height=600
)

fig.show()


---
### Key Takeaways for Students
1. **Embeddings capture semantics:** Unlike BM25, embeddings know that a *"puppy"* is related to a *"dog"* even though they share zero letters.
2. **Vector Space is Geometric:** Closeness in vector space indicates similarity in human concept.
3. **Next Step:** In **Lesson 03**, we learn how to slice long text into **overlapping chunks** and use **Euclidean (L2) distance** to retrieve relevant passages for a user's question!
